# B3 — equilibrium-augmented: does introducing the term LATE avoid the destabilization? (pilot)

**A genuinely different idea, per Omar's explicit instruction to keep trying.** Three independent fixes for items 2/3 have already been tried and falsified: weight recalibration (1.0 -> 0.12), a weight/noise sweep (0.03-0.12 x n_virtual 8/32), and decaying the term OUT starting near the observed ~2,500-iteration onset. The decay pilot's own result was decisive: removing the term early does not prevent the damage, only DELAYS it -- `energy` degrades again roughly 2,000 iterations after the term is fully gone, about matching the original exposure window's own length. All three fixes shared one thing in common: the equilibrium residual was present from **iteration 1** onward.

**This pilot tests the opposite structural idea, never exercised by any of the three prior fixes**: introduce the term **LATE**, onto a network that has already converged far past the destabilization onset via the pure energy loss alone, rather than tuning its weight or phasing it out early. If the damage is specifically caused by perturbing an actively-converging, fragile EARLY-training trajectory, a late introduction (onto an already-converged, more robust network) might behave differently. If the damage is tied to the term itself regardless of when it appears, the same destabilization should reoccur after introduction. Either outcome is a real, reportable finding -- this is not assumed to work going in.

**Two variants, 12,000 iterations each** (short, much cheaper than a full 50,000-iteration run):
- **A. Pure-energy reference** (`w_equilibrium=0` throughout) -- the control, exactly like the original pilot's own reference.
- **B. Late introduction** (`--w_equilibrium_introduce_start 10000 --w_equilibrium_introduce_end 10500`): `w_equilibrium=0` up to iteration 10,000 (4x past the ~2,500 onset), then a quick linear ramp to its full value (0.12) by 10,500, held constant for the remaining 1,500 iterations -- a post-introduction observation window comparable to the original pilot's own (iteration 2,500-4,000, also ~1,500 iterations).

Both use the SAME deterministic `sample_batch(seed_offset=it*batch_size)` sequence (iteration-indexed, not run-indexed), so their `energy` values are directly comparable iteration-by-iteration -- the same methodology the original 4-variant pilot used to establish the ~2,500 onset in the first place. New schedule function `w_equilibrium_late_introduction_at` (in `train_B3_equilibrium_augmented.py`) is CPU-unit-tested (`test_w_equilibrium_late_introduction.py`, 4 checks, all passing) before this notebook.

In [ ]:
# =====================================================================
#  CELL -- Items 2/3 pilot: does a LATE introduction of the weak-form
#  equilibrium residual avoid the destabilization the decay pilot found
#  is otherwise only delayed, not prevented? Two short (12,000-iteration)
#  variants: a pure-energy reference, and late introduction starting at
#  iteration 10,000. See the markdown cell above for the full rationale.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()

print(f'\n{"=" * 90}')
print('=== Re-running the CPU schedule-function correctness checks in this environment ===')
from omar_pfem.test_w_equilibrium_late_introduction import (
    test_disabled_by_default_returns_zero_always, test_zero_before_introduce_start,
    test_full_weight_at_and_after_introduce_end, test_linear_ramp_at_midpoint,
)
test_disabled_by_default_returns_zero_always()
test_zero_before_introduce_start()
test_full_weight_at_and_after_introduce_end()
test_linear_ramp_at_midpoint()

from omar_pfem.train_B3_equilibrium_augmented import get_eq_args, train

device = torch.device('cuda')
R = '/content/drive/MyDrive/pfem_run'

VARIANTS = [
    ("A_pure_energy_reference", ['--w_equilibrium', '0.0']),
    ("B_late_introduction", ['--w_equilibrium', '0.12',
                             '--w_equilibrium_introduce_start', '10000',
                             '--w_equilibrium_introduce_end', '10500']),
]
N_ITERS = 12000

results = {}
for name, extra_args in VARIANTS:
    print(f'\n{"=" * 90}')
    print(f'=== Variant {name} ===')
    OUTPUT_DIR = f'{R}/b3_equilibrium_late_intro_pilot/{name}'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    args = get_eq_args([
        '--n_iters', str(N_ITERS),
        '--batch_size', '8',
        '--log_every', '500',
        '--ckpt_every', str(N_ITERS),
        '--output_dir', OUTPUT_DIR,
        '--n_virtual', '8',
    ] + extra_args)
    print(f'Config: w_equilibrium={args.w_equilibrium}, '
          f'introduce_start={getattr(args, "w_equilibrium_introduce_start", 0)}, '
          f'introduce_end={getattr(args, "w_equilibrium_introduce_end", 0)}, '
          f'n_iters={args.n_iters}')
    torch.cuda.reset_peak_memory_stats(device)
    t0 = time.time()
    model = train(args, device)
    elapsed = time.time() - t0
    print(f'Variant {name} finished in {elapsed:.1f}s ({elapsed / N_ITERS:.4f}s/iteration).')
    results[name] = {'elapsed_s': elapsed, 'output_dir': OUTPUT_DIR}

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(f'{R}/b3_equilibrium_late_intro_pilot', kind='b3_equilibrium_late_intro_pilot',
                    args={'variants': [v[0] for v in VARIANTS], 'n_iters': N_ITERS,
                          'introduce_start': 10000, 'introduce_end': 10500,
                          'w_equilibrium': 0.12},
                    started_at=_started,
                    results={'elapsed_total_s': time.time() - _started,
                             'per_variant': results},
                    outputs=[v['output_dir'] for v in results.values()],
                    notes="Items 2/3 pilot (2026-10-26, per Omar's explicit instruction to "
                          "keep trying): tests whether introducing the weak-form equilibrium "
                          "residual LATE (iteration 10,000, well past the ~2,500-iteration "
                          "onset the earlier 4-variant pilot found) avoids the destabilization "
                          "that the decay-it-out fix already showed is otherwise only delayed, "
                          "not prevented. A genuinely different structural idea from the three "
                          "already-falsified fixes (recalibration, weight/noise sweep, "
                          "decay-out), none of which changed WHEN the term was introduced.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print(f'\n{"=" * 90}')
print(f'Total wall-clock: {time.time() - _started:.1f}s')
print('\nDone. Send this cell\'s full printed output back -- especially variant B\'s own '
      '"energy" trajectory from iteration ~9500 (just before introduction) through 12000 '
      '(1500 iterations after full weight is reached): does it track variant A\'s own '
      'reference trajectory, or diverge upward the same way the original early-introduction '
      'variants did from iteration ~2500 onward?')
